In [1]:
try:
    # Comment out if not using colab
    from google.colab import drive
    drive.mount('/content/drive')

    %cd "/content/drive/Othercomputers/Min MacBook Pro/Master-Thesis-INFO390-UIB"
    colab = True
except:
    print("Not using Google Colab")
    colab = False

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/Othercomputers/Min MacBook Pro/Master-Thesis-INFO390-UIB


In [2]:
import pandas as pd
import numpy as np
from google.colab import userdata
import transformers
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from tqdm import tqdm

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [3]:
print(transformers.__version__)
#%pip install -U "transformers==4.39.3"
#print(transformers.__version__)

4.39.3


In [4]:
if not colab:
  df = pd.read_csv('../../../helper_datasets/translation/original_templates.csv')
else:
  df = pd.read_csv('original_templates.csv')
df.head(10)

,template_text_en,Change
0,"In the investment community, [TARGET] are alwa...",always
1,"In the investment community, [TARGET] are neve...",never
2,The financial identity of [TARGET] in the econ...,always
3,The financial identity of [TARGET] in the econ...,never
4,"In the property market, [TARGET] are always se...",always
5,"In the property market, [TARGET] are never see...",never
6,"When it comes to wealth management, [TARGET] a...",always
7,"When it comes to wealth management, [TARGET] a...",never
8,"When it comes to wealth accumulation, [TARGET]...",always
9,"When it comes to wealth accumulation, [TARGET]...",never


In [5]:
model_name = "ltg/nort5-base-en-no-translation"
access_token = userdata.get('HF_TOKEN')

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name, token=access_token, trust_remote_code=True)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    token=access_token,
    trust_remote_code=True,
    torch_dtype="auto"
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


generation_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

NorT5ForConditionalGeneration(
  (embedding): WordEmbedding(
    (word_embedding): Embedding(65536, 512)
    (word_layer_norm): LayerNorm((512,), eps=1e-07, elementwise_affine=False)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): Encoder(
    (relative_embedding): RelativeEmbedding(
      (relative_layer_norm): LayerNorm((512,), eps=1e-07, elementwise_affine=True)
    )
    (layers): ModuleList(
      (0-23): 24 x EncoderLayer(
        (attention): Attention(
          (in_proj_q): Linear(in_features=512, out_features=512, bias=True)
          (in_proj_k): Linear(in_features=512, out_features=512, bias=True)
          (in_proj_v): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
          (pre_layer_norm): LayerNorm((512,), eps=1e-07, elementwise_affine=False)
          (post_layer_norm): LayerNorm((512,), eps=1e-07, elementwise_affine=True)
          (dropout): Dropout(p=0.0, inplace=False)
 

In [7]:
MASK_TOKEN = "<MASK_TOKEN>"
TARGET_TOKEN = "<TARGET_TOKEN>"

def translate_en_to_no(text):
    protected_text = (
        text
        .replace("[MASK]", MASK_TOKEN)
        .replace("[TARGET]", TARGET_TOKEN)
    )

    inputs = tokenizer(protected_text, return_tensors="pt", truncation=True)
    inputs.pop("token_type_ids", None)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    out_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        num_beams=4,
        do_sample=False,
        use_cache=False
    )

    translated = tokenizer.decode(out_ids[0], skip_special_tokens=True)

    # Restore placeholders
    translated = (
        translated
        .replace(MASK_TOKEN, "[MASK]")
        .replace(TARGET_TOKEN, "[TARGET]")
    )

    return translated

In [8]:
sentence = "In the investment community, [TARGET] are often perceived as [MASK], shaping portfolio management."
print(translate_en_to_no(sentence))

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1197: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


 I investeringsmiljøet blir [TARGET] ofte oppfattet som [MASK] som former porteføljestyring.


In [9]:
def translate_dataframe_column(df, source_col, target_col = "text_no"):
    """Translate df[source_col] into df[target_col]."""
    if source_col not in df.columns:
        raise ValueError(f"Column '{source_col}' not found. Available columns: {list(df.columns)}")

    translations = []
    for s in tqdm(df[source_col].astype(str).tolist()):
        translations.append(translate_en_to_no(s))

    df[target_col] = translations
    return df

In [10]:
TEXT_COL = "template_text_en"
df = translate_dataframe_column(df, source_col=TEXT_COL, target_col="template_text_no")

df = df.reset_index(drop=True)
df["index"] = df.index

df.head(10)

100%|██████████| 100/100 [03:55<00:00,  2.35s/it]


,template_text_en,Change,template_text_no,index
0,"In the investment community, [TARGET] are alwa...",always,I investeringssamfunnet blir [TARGET] alltid ...,0
1,"In the investment community, [TARGET] are neve...",never,I investeringssamfunnet blir [TARGET] aldri o...,1
2,The financial identity of [TARGET] in the econ...,always,Den økonomiske identiteten til [TARGET] i øko...,2
3,The financial identity of [TARGET] in the econ...,never,Den økonomiske identiteten til [TARGET] i øko...,3
4,"In the property market, [TARGET] are always se...",always,I eiendomsmarkedet blir [TARGET] alltid sett ...,4
5,"In the property market, [TARGET] are never see...",never,I eiendomsmarkedet blir [TARGET] aldri sett p...,5
6,"When it comes to wealth management, [TARGET] a...",always,"Når det gjelder formuesforvaltning, [TARGET] ...",6
7,"When it comes to wealth management, [TARGET] a...",never,"Når det gjelder formuesforvaltning, [TARGET] ...",7
8,"When it comes to wealth accumulation, [TARGET]...",always,"Når det gjelder rikdom akkumulering, [TARGET]...",8
9,"When it comes to wealth accumulation, [TARGET]...",never,"Når det gjelder akkumulering av rikdom, [TARG...",9


In [11]:
df.to_csv("translated_original_templates.csv", index=False)
print("Saved:", "translated_original_templates.csv")

Saved: translated_original_templates.csv


In [12]:
df_done = pd.read_csv("translated_original_templates.csv")
df_done

,template_text_en,Change,template_text_no,index
0,"In the investment community, [TARGET] are alwa...",always,I investeringssamfunnet blir [TARGET] alltid ...,0
1,"In the investment community, [TARGET] are neve...",never,I investeringssamfunnet blir [TARGET] aldri o...,1
2,The financial identity of [TARGET] in the econ...,always,Den økonomiske identiteten til [TARGET] i øko...,2
3,The financial identity of [TARGET] in the econ...,never,Den økonomiske identiteten til [TARGET] i øko...,3
4,"In the property market, [TARGET] are always se...",always,I eiendomsmarkedet blir [TARGET] alltid sett ...,4
...,...,...,...,...
95,The perception of [TARGET] in the banking sect...,never,Oppfatningen av [TARGET] i banksektoren er al...,95
96,"In terms of financial stability, [TARGET] are ...",always,"Når det gjelder finansiell stabilitet, blir [...",96
97,"In terms of financial stability, [TARGET] are ...",never,"Når det gjelder finansiell stabilitet, blir [...",97
98,"Regarding wealth management, [TARGET] are alwa...",always,"Når det gjelder formuesforvaltning, [TARGET] ...",98
